## PCA Mutual Information Analysis

**Overview**
- Mutual Information analysis for the top 15 PCA components from the handwriting BCI data set. 
- This code corrects for the bias from a finite data set by plotting the MI calculated from the full, half, quarter, and fifth of the data set (points are randomly selected but maintains the distribution of target characters).
- From these points, we draw a line using linear regression where the y intercept is our estimated MI value if given an infinite data set.

This code was written by Rob Lamprecht.

_Note: Github Copilot powered by Claude 3.5 and Claude 3.7 sonnet was used to assist in the generation of this code. Relevant prompts are included beaneath the cells where AI was used_

**Install Dependencies**

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.feature_selection import mutual_info_classif
import scipy.io as sio
from scipy import stats as st
import statsmodels.api as sm 

**Load data from preprocessed .mat files**  
Some more preprocessing is done to get the arrays to only contain the data relevant to the MI calculations

In [ ]:
# Load in Mat files that contain the target character arrays and an array of PCA values
PCAvalues = sio.loadmat('MIdatasets\PCAvaluesforMI.mat')
TargetChar = sio.loadmat('MIdatasets\letteridsforMI.mat')

In [ ]:
# Select the PCA data needed for MI analysis from the mat file
PCAvalues = PCAvalues['allData_forMI']
print(PCAvalues)

In [ ]:
# Handle the Target Char mat file
# Example component [array(['l'], dtype='<U1')]
# We just want the character

# Extract character function
def orderCharArray (charArray):
  """
  This function takes in a character array and returns a list of characters.
  """
  TargetChar = []
  for i in range(len(charArray)):
    TargetChar.append(charArray[i][0][0])
  return TargetChar

TargetChar = orderCharArray(TargetChar['letteridsforMI'])

print(TargetChar)

# convert target characters to regular strings for the classifier
TargetChar = np.array([str(x) for x in TargetChar])

print(TargetChar)

_Note: Claude was used here._  
Prompt:
" I have a nested array where each value is np.str__('a') for some character, I just want the character"

**Create Helper Functions for MI analyis and Extrapolation**

In [ ]:
def create_subset(X, y, fraction):
    """
    Create a random subset of the data. 
    """
    # Randomly select a subset of the data
    n_samples = int(len(y) * fraction) # Note that this does mean that it will round down
    subset_indices = np.random.choice(len(y), size=n_samples, replace=False)
    
    # Return the selected subset
    X_subset = X[subset_indices]
    y_subset = y[subset_indices]
    
    return X_subset, y_subset

In [ ]:
def mutual_information_for_subset(X, y, fraction, num_sets=10):
    """
    Calculate mutual information for a random subset of the data.
    """
    n_features = X.shape[1]
    mi_values = np.zeros((num_sets, n_features))

    for i in range(num_sets):
        # Create a random subset of the data
        X_subset, y_subset = create_subset(X, y, fraction)
        
        # Calculate MI for each feature independently
        for j in range(n_features):
            feature_data = X_subset[:, j].reshape(-1, 1)  # Reshape to 2D array
            mi_values[i, j] = mutual_info_classif(feature_data, y_subset, discrete_features=False)[0]
    
    return mi_values

_Note: Claude was used in this section for debugging_

"Where is the bug in this block of code? I found that the mi_values that are being saved are the same for every feature" 

**Calculate MI for each subset size**

In [ ]:
# Define the fractions to test
fractions = [.5, .33, .25, .2]

# Define the terms to use for the MI analysis
n = len(PCAvalues)
print(n) # Sanity check verifying the number of samples in the PCA values

x_values = [2/n, 3/n, 4/n, 5/n]

n_feature = PCAvalues.shape[1] # Number of features in the PCA values
print(n_feature) # Sanity check verifying the number of features in the PCA values


In [ ]:
num_sets = 10 # Number of random sets to use for MI analysis

# 3D array where each row is a different fraction, each column is a different random set, and each depth is a different feature
mi_values = np.zeros((len(fractions), num_sets, n_feature))

for i,fraction in enumerate(fractions):
    # Get MI values for this fraction
    mi = mutual_information_for_subset(PCAvalues, TargetChar, fraction, num_sets=10)
    mi_values[i] = mi

_Note: Claude was used here_  

Prompts:  
"Numpy array syntax for three dimensional array"

**Calculate Mean and Standard Error For Each subset size across all components**

In [ ]:
# Calculate means and std errors across subsets
# Axis 1 is the random sets, so we take the mean across that axis
mi_means = np.mean(mi_values, axis=1)
mi_errors = np.std(mi_values, axis=1) / np.sqrt(num_sets)

_Note: Calude was used here_  
Prompts:  
"Numpy operators for mean, standard deviation and squareroot"

**For reference, calculate MI using the full data set**  
Represents MI before bias correction

In [ ]:
def calculate_mi_full(X, y):
    """
    Calculate the mutual information for the full dataset
    """
    n_features = X.shape[1]
    mi_values = np.zeros(n_features)

    mi_values = mutual_info_classif(X, y)
    
    return mi_values


# Calculate MI for the full dataset
# Note: This is not a random subset, but the full dataset
# We'll reference this as full MI or Calculated MI
fullMI = calculate_mi_full(PCAvalues, TargetChar)
print("Full MI values shape:", fullMI.shape)
print(fullMI)

**Plot MI means and run a weighted linear regression**

In [ ]:
# Initialize arrays for results section
extrapolated_mi_values = np.zeros(n_feature)
extrapolated_mi_errors = np.zeros(n_feature)
slopes = np.zeros(n_feature)   
r_squared_values = np.zeros(n_feature)

In [ ]:
# Create a figure with subplots for each feature
n_rows = int(np.ceil(n_feature/3))  # 3 columns
plt.figure(figsize=(n_rows * 4, n_rows * 4))

for feature_idx in range(n_feature):
    
    plt.subplot(n_rows, 3, feature_idx+1)  # Adjust the grid size as needed
    
    # Extract data for this specific feature
    feature_mi_means = mi_means[:, feature_idx]
    feature_mi_errors = mi_errors[:, feature_idx]
    
    # Weights are inverse of squared standard errors
    weights = 1/(feature_mi_errors**2)
    
    # Add constant for intercept term
    X = sm.add_constant(x_values)
    
    # Weighted least squares regression
    wls_model = sm.WLS(feature_mi_means, X, weights=weights)
    results = wls_model.fit()
    
    # Calculate extended x range for regression line
    x_extended = np.linspace(-0.002, max(x_values)*1.1, 100)
    X_extended = sm.add_constant(x_extended)
    regression_line_extended = results.predict(X_extended)
    
    # Calculate y-axis limits with padding
    y_min = min(min(feature_mi_means - feature_mi_errors), results.params[0]) * 0.95
    y_max = max(max(feature_mi_means + feature_mi_errors), results.params[0]) * 1.05
    
    # Plot the data points with error bars
    plt.errorbar(x_values, feature_mi_means, yerr=feature_mi_errors, fmt='o', capsize=5, 
                label='MI values', color='royalblue', markersize=8)
    
    # Plot the extended weighted regression line
    plt.plot(x_extended, regression_line_extended, 'crimson', label='Weighted Regression', linewidth=2)

    # Save the extrapolated MI value and error
    extrapolated_mi_values[feature_idx] = results.params[0]
    extrapolated_mi_errors[feature_idx] = results.bse[0]
    slopes[feature_idx] = results.params[1]
    r_squared_values[feature_idx] = results.rsquared     
    
    # Plot the extrapolated point with error bar
    plt.errorbar([0], [results.params[0]], yerr=[results.bse[0]], fmt='*', color='forestgreen',
                markersize=15, capsize=5, capthick=2, label='Extrapolated MI')
    
    # Plot the full MI value as a dashed line
    plt.axhline(y=fullMI[feature_idx], color='black', linestyle='--', label='Original MI Calculation')
    
    # Set axis limits to show extrapolation clearly
    plt.xlim(-0.001, max(x_values)*1.1)
    plt.ylim(y_min, y_max)
    
    # Add grid
    plt.grid(True, linestyle='--', alpha=0.7)
    
    # Add labels and title
    plt.xlabel('K/N for K = 2, 3, 4, 5')
    plt.ylabel('MI Estimate (bits)')
    plt.title(f'PCA Component {feature_idx+1}: MI Extrapolation')
    
    # Add statistics text
    stats_text = f"MI₀ = {results.params[0]:.4f} ± {results.bse[0]:.4f}\n"
    stats_text += f"Slope = {results.params[1]:.4f}\n"
    stats_text += f"R² = {results.rsquared:.4f}"
    plt.annotate(stats_text, xy=(0.95, 0.05), xycoords='axes fraction',
                bbox=dict(boxstyle="round,pad=0.3", fc="white", alpha=0.8, ec="gray"),
                ha='right', va='bottom')
    
    # Add legend only for the first subplot to avoid clutter
    if feature_idx == 0:
        plt.legend(loc='upper left', fontsize=10, frameon=True, fancybox=True)

    

# Adjust spacing between subplots
plt.tight_layout(pad=3.0)
plt.savefig('Mutual_Information_extrapolation.png', dpi=300, bbox_inches='tight')
plt.show()

_Note: Claude was used extensively Here_  
Prompts:  
"How can I implement a weighted linear regression in python? It does not appear sklearn has support for that in their regression model"  
"display stats text in corner of the plot"
"Error bars in matplot lib"  
"Add grid"  
"Horizontal dashed line in matplot lib"  
"Remind me of subplot syntax in matlab, having an issue with spacing in my current code"  
"How can I save a matplot lib figure with higher resolution?"  
"Multiple versions of green?"

**Additional Visualizations**  
- Bar graph
- Table

In [ ]:
# Create a bar plot comparing original MI vs extrapolated MI with error bars
plt.figure(figsize=(15, 6))
x = np.arange(n_feature)
width = 0.35

plt.bar(x - width/2, fullMI, width, color = 'royalblue',label='Calculated MI (full dataset)')
plt.bar(x + width/2, extrapolated_mi_values, width, color = 'crimson', yerr=extrapolated_mi_errors,
        label='Extrapolated MI (n→∞)', capsize=5)

# Indicate with a '*' if the fullMI value is outside the error bar of the extrapolated MI
for i in range(n_feature):
    if fullMI[i] < (extrapolated_mi_values[i] - extrapolated_mi_errors[i]) or fullMI[i] > (extrapolated_mi_values[i] + extrapolated_mi_errors[i]):
        plt.text(x[i], .55, '*', fontsize=24, ha='center', color='black')

plt.xlabel('PCA Component')
plt.ylabel('Mutual Information')
plt.title('Calculated MI vs Extrapolated MI')
plt.xticks(x, [f'{i+1}' for i in x])
plt.legend()
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.savefig('mi_comparison.png', dpi=300)
plt.show()

_Note: Claude was used here_  
"Remind me of bargraph syntax in matplotlib"   
"Are there multipe versions of red and blue in matplotlib"  


In [ ]:
# Create a table to compare original MI vs extrapolated MI with error bars
# Components should be sorted by the extrapolated MI value
sorted_indices = np.argsort(extrapolated_mi_values)[::-1]  # Sort in descending order
sorted_extrapolated_mi_values = extrapolated_mi_values[sorted_indices]
sorted_extrapolated_mi_errors = extrapolated_mi_errors[sorted_indices]
sorted_fullMI = fullMI[sorted_indices]
sorted_slopes = slopes[sorted_indices]
sorted_r_squared_values = r_squared_values[sorted_indices]
sorted_components = [f'{i+1}' for i in sorted_indices]

fig, ax = plt.subplots(figsize=(10, 5))
# Hide axis
ax.axis('off')

# Prepare data for table
table_data = []
headers = ['Component','Calculated MI', 'Extrapolated MI']
rowlabels = [f'#{i+1}' for i in range(n_feature)]


for i in range(n_feature):
    table_data.append([
        f'Component {sorted_components[i]}',
        f'{sorted_fullMI[i]:.4f}',
        f'{sorted_extrapolated_mi_values[i]:.4f} ± {sorted_extrapolated_mi_errors[i]:.4f}'
        
    ])


# Create the table
table = ax.table(
    cellText=table_data,
    colLabels=headers,
    rowLabels=rowlabels,
    cellLoc='center',
    loc='center',
    colColours=['lightgray', 'royalblue', 'crimson'],
    rowColours=['lightgray'] * n_feature,
    cellColours=[['white'] * len(headers) for _ in range(n_feature)],
    colWidths= [.1, .15, .15],
)

# Style the table
table.auto_set_font_size(False)
table.set_fontsize(9)
table.scale(1.2, 1.5)  # Make cells bigger

# Add a title
plt.title('PCA components ranked by Extrapolated MI ', pad=20)

plt.tight_layout()
plt.savefig('mi_table.png', dpi=300, bbox_inches='tight')
plt.show()

_Note: Claude was used here_  
Prompts:  
"How to make a display table in matplotlib"
